# Penelope v3 ambiguity demo

This notebook demonstrates side-by-side behavior for v3 SFT and v3 GRPO on:
- clear SoC passages
- clear non-SoC passages
- ambiguity-heavy passages that could plausibly map to multiple SoC types

It highlights whether each model signals ambiguity using `soc_type`, `secondary_devices`, `confidence`, `notes`, and `explanation`, and adds uncertainty-aware diagnostics plus exportable outputs.

In [ ]:
from __future__ import annotations

import json
import random
import re
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

from scripts.prompts import TRAINING_SYSTEM_PROMPT

pd.set_option('display.max_colwidth', 220)

BASE_MODEL_CANDIDATE = Path('/scratch/gpfs/MM4/apjanco/.cache/huggingface/hub/models--Qwen--Qwen3-4B/snapshots/1cfa9a7208912126459214e8b04321603b3df60c')
BASE_MODEL = str(BASE_MODEL_CANDIDATE) if BASE_MODEL_CANDIDATE.exists() else 'Qwen/Qwen3-4B'

SFT_ADAPTER_PATH = Path('adapter-v3')
GRPO_CKPT_CANDIDATE = Path('adapter-v3-grpo/checkpoints/checkpoint-5291')
GRPO_ADAPTER_PATH = GRPO_CKPT_CANDIDATE if GRPO_CKPT_CANDIDATE.exists() else Path('adapter-v3-grpo')

MODEL_SPECS = {
    'v3_sft': {'label': 'v3_sft', 'adapter_path': SFT_ADAPTER_PATH},
    'v3_grpo': {'label': 'v3_grpo', 'adapter_path': GRPO_ADAPTER_PATH},
}
ACTIVE_MODELS = ['v3_sft', 'v3_grpo']

MAX_NEW_TOKENS = 1200
ENABLE_THINKING = True
TEMPERATURE = 0.6
TOP_P = 0.95
TOP_K = 20


def resolve_runtime_config() -> dict[str, Any]:
    if not torch.cuda.is_available():
        return {
            'device_map': 'cpu',
            'torch_dtype': torch.float32,
            'attn_implementation': 'eager',
            'runtime_note': 'CUDA not available; using CPU inference.',
            'cuda_usable': False,
        }

    supported_arches = {arch.replace('sm_', '') for arch in torch.cuda.get_arch_list()}
    device_props = torch.cuda.get_device_properties(0)
    current_arch = f"{device_props.major}{device_props.minor}"
    cuda_usable = current_arch in supported_arches

    if not cuda_usable:
        return {
            'device_map': 'cpu',
            'torch_dtype': torch.float32,
            'attn_implementation': 'eager',
            'runtime_note': (
                f'CUDA device {device_props.name} has compute capability {device_props.major}.{device_props.minor}, '
                f'but this PyTorch build only supports {sorted(supported_arches)}. Falling back to CPU to avoid '
                'cudaErrorNoKernelImageForDevice.'
            ),
            'cuda_usable': False,
        }

    torch_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return {
        'device_map': 'auto',
        'torch_dtype': torch_dtype,
        'attn_implementation': 'sdpa',
        'runtime_note': f'Using CUDA on {device_props.name} (cc {device_props.major}.{device_props.minor}).',
        'cuda_usable': True,
    }


RUNTIME_CONFIG = resolve_runtime_config()
DEVICE_MAP = RUNTIME_CONFIG['device_map']
TORCH_DTYPE = RUNTIME_CONFIG['torch_dtype']
ATTN_IMPLEMENTATION = RUNTIME_CONFIG['attn_implementation']

print(f'BASE_MODEL={BASE_MODEL}')
for mid in ACTIVE_MODELS:
    p = MODEL_SPECS[mid]['adapter_path']
    print(f'{mid}: {p.resolve()} (exists={p.exists()})')
print(RUNTIME_CONFIG['runtime_note'])
print(f'device_map={DEVICE_MAP}, dtype={TORCH_DTYPE}, cuda_available={torch.cuda.is_available()}, cuda_usable={RUNTIME_CONFIG["cuda_usable"]}')

/home/aj7878/.conda/envs/penelope/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BASE_MODEL=/scratch/gpfs/MM4/apjanco/.cache/huggingface/hub/models--Qwen--Qwen3-4B/snapshots/1cfa9a7208912126459214e8b04321603b3df60c
v3_sft: /scratch/gpfs/MM4/apjanco/penelope/adapter-v3 (exists=True)
v3_grpo: /scratch/gpfs/MM4/apjanco/penelope/adapter-v3-grpo/checkpoints/checkpoint-5291 (exists=True)
dtype=torch.bfloat16, cuda=True


In [ ]:
def load_model_bundle(model_id: str, base_model: str = BASE_MODEL):
    if model_id not in MODEL_SPECS:
        raise KeyError(f'Unknown model_id: {model_id}')

    adapter_path = Path(MODEL_SPECS[model_id]['adapter_path'])
    if not adapter_path.exists():
        raise FileNotFoundError(f'Adapter path does not exist for {model_id}: {adapter_path}')

    tokenizer = AutoTokenizer.from_pretrained(str(adapter_path), trust_remote_code=True)
    base = AutoModelForCausalLM.from_pretrained(
        base_model,
        torch_dtype=TORCH_DTYPE,
        device_map=DEVICE_MAP,
        trust_remote_code=True,
        attn_implementation=ATTN_IMPLEMENTATION,
    )
    model = PeftModel.from_pretrained(base, str(adapter_path))
    model.eval()
    return tokenizer, model

print('Loader ready. Models are loaded lazily during inference.')

Loader ready. Models are loaded lazily during inference.


In [3]:
def build_messages(passage: str, source: str) -> list[dict[str, str]]:
    user = (
        f'Source: {source}\n'
        'Chunk: demo-0 (notebook-demo)\n'
        'Position: 0\n\n'
        '[TEXT TO ANALYZE]\n'
        f'{passage}\n\n'
        'Analyze the [TEXT TO ANALYZE] section above. '
        'Identify and classify every SoC passage. '
        'Respond with JSON only.'
    )
    return [
        {'role': 'system', 'content': TRAINING_SYSTEM_PROMPT},
        {'role': 'user', 'content': user},
    ]


def run_inference(model, tokenizer, messages: list[dict[str, str]]) -> str:
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING,
    )
    inputs = tokenizer(text, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE if ENABLE_THINKING else 0.7,
            top_p=TOP_P if ENABLE_THINKING else 0.8,
            top_k=TOP_K if ENABLE_THINKING else 20,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


def parse_response(raw: str) -> dict[str, Any]:
    raw_clean = raw.strip()
    think_match = re.search(r'<think>(.*?)</think>', raw_clean, re.DOTALL)
    think_text = think_match.group(1).strip() if think_match else ''

    json_candidate = re.sub(r'<think>.*?</think>', '', raw_clean, flags=re.DOTALL).strip()
    json_match = re.search(r'\{.*\}', json_candidate, re.DOTALL)
    if not json_match:
        return {'instances': [], '_think': think_text, '_raw': raw_clean}

    try:
        parsed = json.loads(json_match.group(0))
    except json.JSONDecodeError:
        parsed = {'instances': []}

    if 'instances' not in parsed or not isinstance(parsed['instances'], list):
        parsed['instances'] = []

    parsed['_think'] = think_text
    parsed['_raw'] = raw_clean
    return parsed


def top_instance(parsed: dict[str, Any]) -> dict[str, Any]:
    instances = parsed.get('instances', [])
    if isinstance(instances, list) and len(instances) > 0 and isinstance(instances[0], dict):
        return instances[0]
    return {}


def normalize_secondary_devices(value: Any) -> list[str]:
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    if isinstance(value, str):
        return [x.strip() for x in value.split(',') if x.strip()]
    return []


def has_multi_type_signal(inst: dict[str, Any]) -> bool:
    secondary = normalize_secondary_devices(inst.get('secondary_devices', []))
    notes = str(inst.get('notes', '')).lower()
    explanation = str(inst.get('explanation', '')).lower()

    hedge_terms = ['hybrid', 'ambiguous', 'could', 'also', 'overlap', 'mixed', 'alternative']
    hedged = any(term in notes or term in explanation for term in hedge_terms)
    return bool(secondary) or hedged


CONFIDENCE_MAP = {'high': 0.9, 'medium': 0.7, 'low': 0.55}
NON_SOC_CONFIDENCE_DEFAULT = 0.65


def prediction_confidence(pred_is_soc: bool, inst: dict[str, Any]) -> float:
    if pred_is_soc:
        return float(CONFIDENCE_MAP.get(str(inst.get('confidence', 'medium')).lower(), 0.7))
    return float(NON_SOC_CONFIDENCE_DEFAULT)


def compute_ece(correct: np.ndarray, conf: np.ndarray, n_bins: int = 10) -> float:
    if len(correct) == 0:
        return 0.0
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i < n_bins - 1:
            idx = (conf >= lo) & (conf < hi)
        else:
            idx = (conf >= lo) & (conf <= hi)
        if not np.any(idx):
            continue
        acc_bin = float(np.mean(correct[idx]))
        conf_bin = float(np.mean(conf[idx]))
        w = float(np.mean(idx))
        ece += w * abs(acc_bin - conf_bin)
    return float(ece)


def compute_risk_coverage(correct: np.ndarray, conf: np.ndarray) -> tuple[pd.DataFrame, float]:
    if len(correct) == 0:
        return pd.DataFrame(columns=['coverage', 'risk']), 0.0

    order = np.argsort(-conf)
    c_sorted = correct[order]

    coverages = []
    risks = []
    for k in range(1, len(c_sorted) + 1):
        coverage = k / len(c_sorted)
        risk = 1.0 - float(np.mean(c_sorted[:k]))
        coverages.append(coverage)
        risks.append(risk)

    curve = pd.DataFrame({'coverage': coverages, 'risk': risks})
    aurc = float(np.trapz(curve['risk'].values, curve['coverage'].values))
    return curve, aurc

In [4]:
# Curated diagnostic passages.
CURATED_PASSAGES = [
    {
        'id': 'soc_direct_1',
        'source': 'Curated direct interior',
        'kind': 'clear_soc',
        'gold_is_soc': True,
        'gold_primary': 'direct_interior_monologue',
        'gold_alt_types': ['free_association'],
        'text': 'I cannot keep the thought still, not one of them, they run ahead of me and double back, and if I close my eyes I hear the bell again, no, not a bell, the spoon against the glass, and why did I say yes, why did I nod, foolish, foolish.',
    },
    {
        'id': 'soc_indirect_1',
        'source': 'Curated indirect interior',
        'kind': 'clear_soc',
        'gold_is_soc': True,
        'gold_primary': 'indirect_interior_monologue',
        'gold_alt_types': ['omniscient_description'],
        'text': 'She crossed the square more quickly than she meant to, for of course he would be there, and of course she would not look, and was that not exactly the sort of weakness she had promised herself she had finished with years ago.',
    },
    {
        'id': 'non_soc_1',
        'source': 'Curated external narration',
        'kind': 'clear_non_soc',
        'gold_is_soc': False,
        'gold_primary': 'none',
        'gold_alt_types': [],
        'text': 'The train arrived at half past six. Three porters unloaded the trunks, stacked them by the station wall, and waited for instructions from the conductor.',
    },
    {
        'id': 'non_soc_2',
        'source': 'Curated descriptive narration',
        'kind': 'clear_non_soc',
        'gold_is_soc': False,
        'gold_primary': 'none',
        'gold_alt_types': [],
        'text': 'Rain moved across the field in narrow bands. The barn roof shone, then dulled, while crows gathered along the fence and settled for the night.',
    },
    {
        'id': 'ambig_hybrid_1',
        'source': 'Curated hybrid candidate',
        'kind': 'ambiguous',
        'gold_is_soc': True,
        'gold_primary': 'hybrid',
        'gold_alt_types': ['free_association', 'imagery', 'simulation_state_of_mind'],
        'text': 'The room turns to water and then to wool, soft and choking, and her name is a thread in my mouth, blue thread, bitter thread, pull it and the whole evening comes apart.',
    },
    {
        'id': 'ambig_reverie_1',
        'source': 'Curated reverie candidate',
        'kind': 'ambiguous',
        'gold_is_soc': True,
        'gold_primary': 'reverie_fantasy',
        'gold_alt_types': ['simulation_state_of_mind', 'imagery'],
        'text': 'If I stepped through that doorway now the whole city would fold like paper, and mother would be in the kitchen again, humming, though the smell of smoke says this is no kitchen and no year I can name.',
    },
    {
        'id': 'ambig_indirect_omniscient_1',
        'source': 'Curated indirect-omniscient edge',
        'kind': 'ambiguous',
        'gold_is_soc': True,
        'gold_primary': 'indirect_interior_monologue',
        'gold_alt_types': ['omniscient_description'],
        'text': 'He paused before answering, astonished at how quickly pride dressed itself as caution; perhaps that was always his trick, to call fear prudence and feel wise for it.',
    },
    {
        'id': 'soc_orthographic_1',
        'source': 'Curated orthographic marker',
        'kind': 'clear_soc',
        'gold_is_soc': True,
        'gold_primary': 'orthographic_marker',
        'gold_alt_types': ['direct_interior_monologue'],
        'text': 'No no no -- not again -- if the door clicks I will -- I cannot -- stop -- breathe -- count -- one two three --',
    },
]

pd.DataFrame(CURATED_PASSAGES)[['id', 'kind', 'gold_is_soc', 'gold_primary', 'gold_alt_types']]

,id,kind,gold_is_soc,gold_primary,gold_alt_types
0,soc_direct_1,clear_soc,True,direct_interior_monologue,[free_association]
1,soc_indirect_1,clear_soc,True,indirect_interior_monologue,[omniscient_description]
2,non_soc_1,clear_non_soc,False,none,[]
3,non_soc_2,clear_non_soc,False,none,[]
4,ambig_hybrid_1,ambiguous,True,hybrid,"[free_association, imagery, simulation_state_of_mind]"
5,ambig_reverie_1,ambiguous,True,reverie_fantasy,"[simulation_state_of_mind, imagery]"
6,ambig_indirect_omniscient_1,ambiguous,True,indirect_interior_monologue,[omniscient_description]
7,soc_orthographic_1,clear_soc,True,orthographic_marker,[direct_interior_monologue]


## Optional: add real labeled passages from dataset/test_raw.jsonl

In [5]:
REAL_DATA_PATH = Path('dataset/test_raw.jsonl')
USE_REAL_SAMPLES = True
N_REAL_SOC = 3
N_REAL_NON_SOC = 3
SEED = 13

def load_real_samples(path: Path, n_soc: int = 3, n_non_soc: int = 3, seed: int = 13) -> list[dict[str, Any]]:
    if not path.exists():
        return []

    records = []
    with path.open('r', encoding='utf-8') as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                continue

    soc = [r for r in records if bool(r.get('is_soc'))]
    non_soc = [r for r in records if not bool(r.get('is_soc'))]

    rng = random.Random(seed)
    if len(soc) > n_soc:
        soc = rng.sample(soc, n_soc)
    if len(non_soc) > n_non_soc:
        non_soc = rng.sample(non_soc, n_non_soc)

    out = []
    for i, rec in enumerate(soc + non_soc):
        gold_soc = bool(rec.get('is_soc'))
        out.append({
            'id': f'real_{i:02d}',
            'source': str(rec.get('source_file', 'dataset/test_raw.jsonl')),
            'kind': 'real_soc' if gold_soc else 'real_non_soc',
            'gold_is_soc': gold_soc,
            'gold_primary': 'unknown',
            'gold_alt_types': [],
            'text': str(rec.get('passage', ''))[:3000],
        })
    return out

real_samples = load_real_samples(REAL_DATA_PATH, N_REAL_SOC, N_REAL_NON_SOC, SEED) if USE_REAL_SAMPLES else []
print(f'Loaded real samples: {len(real_samples)}')
pd.DataFrame(real_samples)[['id', 'kind', 'gold_is_soc', 'source']].head() if real_samples else None

Loaded real samples: 6


,id,kind,gold_is_soc,source
0,real_00,real_soc,True,the_years.txt
1,real_01,real_soc,True,pg01399_anna_karenina.txt
2,real_02,real_soc,True,pg04300_ulysses.txt
3,real_03,real_non_soc,False,pg30059_the_wings_of_the_dove_volume_2.txt
4,real_04,real_non_soc,False,pg00209_the_turn_of_the_screw.txt


In [6]:
PASSAGES = CURATED_PASSAGES + real_samples
print(f'Total passages to run: {len(PASSAGES)}')

Total passages to run: 14


In [7]:
rows = []

for model_id in ACTIVE_MODELS:
    print(f'Running model: {model_id}')
    tokenizer, model = load_model_bundle(model_id)

    for p in PASSAGES:
        messages = build_messages(p['text'], p['source'])
        raw = run_inference(model, tokenizer, messages)
        parsed = parse_response(raw)
        inst = top_instance(parsed)

        pred_instances = parsed.get('instances', [])
        pred_is_soc = isinstance(pred_instances, list) and len(pred_instances) > 0
        secondary_list = normalize_secondary_devices(inst.get('secondary_devices', []))

        conf_num = prediction_confidence(pred_is_soc, inst)
        correct_binary = bool(pred_is_soc == p['gold_is_soc'])

        rows.append({
            'model_id': model_id,
            'id': p['id'],
            'kind': p['kind'],
            'source': p['source'],
            'gold_is_soc': p['gold_is_soc'],
            'gold_primary': p['gold_primary'],
            'gold_alt_types': ', '.join(p['gold_alt_types']) if p['gold_alt_types'] else '',
            'pred_is_soc': pred_is_soc,
            'pred_type': inst.get('soc_type', 'none') if pred_is_soc else 'none',
            'pred_secondary': ', '.join(secondary_list),
            'pred_confidence': inst.get('confidence', ''),
            'pred_confidence_num': conf_num,
            'correct_binary': correct_binary,
            'pred_explanation': inst.get('explanation', ''),
            'pred_notes': inst.get('notes', ''),
            'multi_type_signal': has_multi_type_signal(inst) if pred_is_soc else False,
            'think_preview': parsed.get('_think', '')[:320],
            'raw_output': parsed.get('_raw', raw),
        })

    del model
    del tokenizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


df = pd.DataFrame(rows)

# Wide comparison table for quick side-by-side inspection.
compare_cols = [
    'pred_is_soc',
    'pred_type',
    'pred_secondary',
    'pred_confidence',
    'pred_confidence_num',
    'multi_type_signal',
]

df_compare = (
    df.set_index(['id', 'kind', 'gold_is_soc', 'gold_primary', 'gold_alt_types', 'model_id'])[compare_cols]
      .unstack('model_id')
)

print(f'Rows generated: {len(df)}')
df_compare

Running model: v3_sft


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
/home/aj7878/.conda/envs/penelope/lib/python3.12/site-packages/torch/cuda/__init__.py:371: UserWarning: Found GPU0 Tesla P100-PCIE-16GB which is of compute capability (CC) 6.0.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Please follow the instructions at https://pytorch.org/get-started/locally/ to install a PyTorch release that supports one of these CUDA versions: 12.6
  _warn_unsupported_code(d, device_cc, code_ccs)
/home/aj7878/.conda/envs/penelope/lib/python3.12/site-packages/torch/cuda/__init__.py:371: UserWarning: Found GPU1 Tesla P100-PCIE-16GB which 

AcceleratorError: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# Binary and uncertainty-aware metrics per model.
eval_df = df[df['gold_is_soc'].isin([True, False])].copy()

summary_rows = []
risk_curves = {}

for model_id, g in eval_df.groupby('model_id'):
    y_true = g['gold_is_soc'].astype(bool).values
    y_pred = g['pred_is_soc'].astype(bool).values

    tp = int(np.sum((y_true == True) & (y_pred == True)))
    tn = int(np.sum((y_true == False) & (y_pred == False)))
    fp = int(np.sum((y_true == False) & (y_pred == True)))
    fn = int(np.sum((y_true == True) & (y_pred == False)))

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    acc = (tp + tn) / len(g) if len(g) else 0.0

    conf = g['pred_confidence_num'].astype(float).values
    correct = g['correct_binary'].astype(float).values

    # Uncertainty proxies based on confidence attached to the predicted label.
    ece = compute_ece(correct, conf, n_bins=10)
    brier = float(np.mean((conf - correct) ** 2))
    curve_df, aurc = compute_risk_coverage(correct, conf)
    curve_df['model_id'] = model_id
    risk_curves[model_id] = curve_df

    summary_rows.append(
        {
            'model_id': model_id,
            'n_cases': int(len(g)),
            'tp': tp,
            'tn': tn,
            'fp': fp,
            'fn': fn,
            'precision': round(precision, 3),
            'recall': round(recall, 3),
            'f1': round(f1, 3),
            'accuracy': round(acc, 3),
            'ece_proxy': round(ece, 4),
            'brier_proxy': round(brier, 4),
            'aurc_proxy': round(aurc, 4),
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values('model_id').reset_index(drop=True)
summary_df

In [ ]:
# Ambiguity-focused view: overlap signal by model.
ambig_view = df[(df['kind'] == 'ambiguous') | (df['multi_type_signal'] == True)].copy()
ambig_view[
    [
        'model_id',
        'id',
        'kind',
        'gold_primary',
        'gold_alt_types',
        'pred_type',
        'pred_secondary',
        'pred_confidence',
        'pred_confidence_num',
        'multi_type_signal',
        'pred_explanation',
        'pred_notes',
    ]
].sort_values(['id', 'model_id'])

In [ ]:
def show_case(case_id: str, model_id: str) -> None:
    hit = df[(df['id'] == case_id) & (df['model_id'] == model_id)]
    if hit.empty:
        print('No case found: {} [{}]'.format(case_id, model_id))
        return

    row = hit.iloc[0]
    print('MODEL: {}'.format(model_id))
    print('CASE: {} ({})'.format(row['id'], row['kind']))
    print('Gold SoC: {} | Pred SoC: {}'.format(row['gold_is_soc'], row['pred_is_soc']))
    print('Gold type: {} | Pred type: {}'.format(row['gold_primary'], row['pred_type']))
    print('Gold alternatives: {}'.format(row['gold_alt_types']))
    print('Pred secondary: {}'.format(row['pred_secondary']))
    print('Confidence: {} ({:.2f})'.format(row['pred_confidence'], row['pred_confidence_num']))
    print('Multi-type signal: {}'.format(row['multi_type_signal']))
    print('\nExplanation:\n', row['pred_explanation'])
    print('\nNotes:\n', row['pred_notes'])
    print('\nThink preview:\n', row['think_preview'])

for cid in ['ambig_hybrid_1', 'ambig_reverie_1', 'ambig_indirect_omniscient_1']:
    print('\n' + '=' * 80)
    for mid in ACTIVE_MODELS:
        show_case(cid, mid)
        print('-' * 40)

In [ ]:
# Optional: inspect full raw output for a single case/model.
CASE_ID = 'ambig_hybrid_1'
MODEL_ID = 'v3_sft'
row = df[(df['id'] == CASE_ID) & (df['model_id'] == MODEL_ID)].iloc[0]
print(row['raw_output'])

In [ ]:
# Export outputs for reporting.
EXPORT_DIR = Path('outputs/notebook_demo')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

run_tag = pd.Timestamp.utcnow().strftime('v3_demo_%Y%m%d_%H%M%S')
raw_csv_path = EXPORT_DIR / f'{run_tag}_predictions.csv'
raw_json_path = EXPORT_DIR / f'{run_tag}_predictions.json'
summary_csv_path = EXPORT_DIR / f'{run_tag}_summary.csv'
summary_json_path = EXPORT_DIR / f'{run_tag}_summary.json'
ambig_csv_path = EXPORT_DIR / f'{run_tag}_ambiguity.csv'

# Long-form predictions (one row per case per model)
df.to_csv(raw_csv_path, index=False)
df.to_json(raw_json_path, orient='records', indent=2)

# Metrics summary
summary_df.to_csv(summary_csv_path, index=False)
summary_df.to_json(summary_json_path, orient='records', indent=2)

# Ambiguity slice
ambig_view.to_csv(ambig_csv_path, index=False)

print('Exported files:')
for p in [raw_csv_path, raw_json_path, summary_csv_path, summary_json_path, ambig_csv_path]:
    print('-', p)

## Notes

- This notebook now runs a direct v3 SFT vs v3 GRPO comparison on the same passages.
- `ece_proxy`, `brier_proxy`, and `aurc_proxy` are confidence-based uncertainty proxies tied to predicted-label confidence.
- Non-SoC predictions use `NON_SOC_CONFIDENCE_DEFAULT`; tune it if you calibrate explicit non-SoC confidence.
- Outputs are exportable to CSV/JSON under outputs/notebook_demo.